In [16]:
# ============================================================
# Local Document Q&A Chatbot using RAG + Ollama
# Single file project. Simple structure.
# Supports PDF, CSV, and TXT documents.
# ============================================================

import os

# Load PDF documents
from langchain_community.document_loaders import PyPDFLoader

# Load CSV documents
from langchain_community.document_loaders import CSVLoader

# Load TXT documents
from langchain_community.document_loaders import TextLoader

# Split long documents into smaller chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Vector database used to store embeddings
from langchain_community.vectorstores import Chroma

# Embedding model used to convert text into vectors
from langchain_community.embeddings import HuggingFaceEmbeddings

# Ollama local LLM
from langchain_community.llms import Ollama

import warnings
warnings.filterwarnings("ignore")

In [17]:

# ============================================================
# Folder paths
# ============================================================

# Folder where documents are stored
DATA_FOLDER = "data"

# Folder where vector database will be saved
VECTOR_DB_FOLDER = "vector_db"

In [18]:
# ============================================================
# Step 1. Load documents from folder
# ============================================================

def load_documents():
    """
    Reads all documents from the data folder.
    Supported formats: PDF, CSV, TXT
    """

    documents = []

    print("Scanning data folder...\n")

    # loop through files inside data folder
    for file in os.listdir(DATA_FOLDER):

        file_path = os.path.join(DATA_FOLDER, file)

        # load PDF file
        if file.endswith(".pdf"):
            print("Loading PDF:", file)

            loader = PyPDFLoader(file_path)
            docs = loader.load()

            documents.extend(docs)

            print("PDF loaded successfully. Pages:", len(docs), "\n")

        # load CSV file
        elif file.endswith(".csv"):
            print("Loading CSV:", file)

            loader = CSVLoader(file_path)
            docs = loader.load()

            documents.extend(docs)

            print("CSV loaded successfully. Rows:", len(docs), "\n")

        # load TXT file
        elif file.endswith(".txt"):
            print("Loading TXT:", file)

            loader = TextLoader(file_path)
            docs = loader.load()

            documents.extend(docs)

            print("TXT loaded successfully. Lines:", len(docs), "\n")

    print("Total documents loaded:", len(documents))

    return documents

docs = load_documents()

Scanning data folder...

Loading PDF: Backtracking.pdf
PDF loaded successfully. Pages: 12 

Loading TXT: notes.txt.txt
TXT loaded successfully. Lines: 1 

Loading CSV: spam.csv
CSV loaded successfully. Rows: 5572 

Total documents loaded: 5585


In [19]:
# ============================================================
# Step 2. Split documents into chunks
# ============================================================

def split_documents(documents):
    """
    Split large documents into smaller chunks.
    This helps the retrieval system search better.
    """

    print("\nStarting document splitting...\n")

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100
    )

    chunks = text_splitter.split_documents(documents)

    print("Documents split successfully.")
    print("Total chunks created:", len(chunks), "\n")

    # show preview of first chunk
    print("Sample chunk preview:\n")
    print(chunks[0].page_content[:300])
    print("\n-----------------------------------\n")

    return chunks

chunks = split_documents(docs)


Starting document splitting...

Documents split successfully.
Total chunks created: 5633 

Sample chunk preview:

1 | P a g e   N o t e  B y :  S a r o j  B h a n d a r i  
 
 
Backtracking  
Introduction 
• Backtracking is a general algorithmic technique for ﬁnding solutions to  some computational 
problems, notably constraint satisfaction problems 
• Backtracking incrementally builds candidates to the solutio

-----------------------------------



In [20]:
# ============================================================
# Step 3. Create vector database using embeddings
# ============================================================

def create_vector_database(chunks):
    """
    Convert text chunks into embeddings
    and store them inside a vector database.
    """

    print("\nStarting vector database creation...\n")

    # embedding model
    print("Loading embedding model...")
    embeddings = HuggingFaceEmbeddings()
    print("Embedding model loaded.\n")

    # create vector database
    print("Creating Chroma vector database...")
    db = Chroma.from_documents(
        chunks,
        embeddings,
        persist_directory=VECTOR_DB_FOLDER
    )

    # save database locally
    print("Saving vector database to disk...")
    db.persist()

    print("Vector database created successfully.")
    print("Total chunks stored:", len(chunks), "\n")

    # preview first chunk stored
    print("Sample stored chunk:\n")
    print(chunks[0].page_content[:300])
    print("\n---------------------------------\n")

    return db

In [21]:

# ============================================================
# Step 4. Load existing vector database
# ============================================================

def load_vector_database():
    """
    Load stored vector database from disk.
    """

    embeddings = HuggingFaceEmbeddings()

    db = Chroma(
        persist_directory=VECTOR_DB_FOLDER,
        embedding_function=embeddings
    )

    return db

db = create_vector_database(chunks)



Starting vector database creation...

Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded.

Creating Chroma vector database...
Saving vector database to disk...
Vector database created successfully.
Total chunks stored: 5633 

Sample stored chunk:

1 | P a g e   N o t e  B y :  S a r o j  B h a n d a r i  
 
 
Backtracking  
Introduction 
• Backtracking is a general algorithmic technique for ﬁnding solutions to  some computational 
problems, notably constraint satisfaction problems 
• Backtracking incrementally builds candidates to the solutio

---------------------------------



In [14]:
# ============================================================
# Step 5. Chatbot interaction loop
# ============================================================

def start_chatbot():
    """
    Start chatbot and answer questions using RAG.
    """

    print("\nStarting chatbot...\n")

    # load vector database
    print("Loading vector database...")
    db = load_vector_database()
    print("Vector database loaded successfully.\n")

    # connect to Ollama model
    print("Connecting to Ollama model llama3...")
    llm = Ollama(model="llama3")
    print("Model connected successfully.\n")

    print("RAG Chatbot ready.")
    print("Type 'exit' to stop.\n")

    while True:

        # user question
        query = input("Ask a question: ")

        # stop condition
        if query.lower() == "exit":
            print("\nChatbot stopped.")
            break

        print("\nSearching relevant documents...\n")

        # retrieve most relevant chunks
        docs = db.similarity_search(query, k=3)

        print("Relevant chunks retrieved:", len(docs), "\n")

        # build context
        context = ""

        for i, doc in enumerate(docs):
            print("Chunk", i + 1, "retrieved\n")
            print(doc.page_content[:300])
            print("\n-----------------------------\n")

            context += doc.page_content + "\n"

        # prompt for model
        prompt = f"""
You are a document question answering assistant.

Rules:
1. Answer only from the provided context.
2. Do not say you cannot access files.
3. Do not copy the context word for word unless needed.
4. Give a short, clear answer.
5. If the answer is not in the context, reply exactly:
Answer not found in documents.

Context:
{context}

Question:
{query}

Final answer:
"""

        print("Generating answer from model...\n")

        # generate response
        response = llm.invoke(prompt)

        print("Answer:\n")
        print(response)
        print("\n====================================\n")

In [15]:
start_chatbot()


Starting chatbot...

Loading vector database...


C:\Users\User\AppData\Local\Temp\ipykernel_11952\3299572284.py:10: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector database loaded successfully.

Connecting to Ollama model llama3...
Model connected successfully.

RAG Chatbot ready.
Type 'exit' to stop.



...


Searching relevant documents...

Relevant chunks retrieved: 3 

Chunk 1 retrieved

v1: ham
v2: I forgot 2 ask Ì_ all smth.. There's a card on da present lei... How? ÌÏ all want 2 write smth or sign on it?
:

-----------------------------

Chunk 2 retrieved

v1: ham
v2: Really... I tot ur paper ended long ago... But wat u copied jus now got use? U happy lar... I still haf 2 study :-(
:

-----------------------------

Chunk 3 retrieved

v1: ham
v2: A bit of Ur smile is my hppnss, a drop of Ur tear is my sorrow, a part of Ur heart is my life, a heart like mine wil care for U, forevr as my GOODFRIEND
:

-----------------------------

Generating answer from model...

Answer:

According to the provided context, the author of the note is "Ur" (a person referred to in the notes).




Ask a question:  What name appears in the footer



Searching relevant documents...

Relevant chunks retrieved: 3 

Chunk 1 retrieved

v1: spam
v2: Jamster! To get your free wallpaper text HEART to 88888 now! T&C apply. 16 only. Need Help? Call 08701213186.
:

-----------------------------

Chunk 2 retrieved

v1: spam
v2: URGENT. Important information for 02 user. Today is your lucky day! 2 find out why , log onto http://www.urawinner.com there is a fantastic surprise awaiting you !
:

-----------------------------

Chunk 3 retrieved

v1: spam
v2: You are being contacted by our Dating Service by someone you know! To find out who it is, call from your mobile or landline 09064017305 PoBox75LDNS7
:

-----------------------------

Generating answer from model...

Answer:

Answer not found in documents.




Ask a question:  exit



Chatbot stopped.


C:\Users\User\AppData\Local\Temp\ipykernel_4144\3359106738.py:10: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



RAG Chatbot ready. Type 'exit' to stop.



Ask a question:  what is in footer of pdf document



Retrieved context:



Answer:
 After searching through the documents, I found that the footer of the PDF document contains the text "Copyright 2023 XYZ Corporation. All rights reserved." This information can be found in a document labeled "PDF Document Properties".


Ask a question:  who wrote the pdf note



Retrieved context:



Answer:
 According to the context, there is a document with the following properties:

```
{
  "_id": "pdf-note",
  "author": "John Smith",
  "content": "This is a PDF note.",
  "format": "PDF"
}
```

Therefore, I can answer that the PDF note was written by John Smith.


Ask a question:  exit
